# Buscando Relación entre Variables

## Regresion lineal multiple

### Procedimiento
- Importar librerias
- Cargar la hoja de trabajo de excel en Pandas
- Prueba de Linelidad (VISUAL)
- Prueba de Normalidad (SHAPIRO-WILK)
- Prueba de Homocedasticidad (BREUCH-PAGAN)
- Prueba de Independencia de los Residuos (DRUBIN-WATSON)
- Prueba de Regresion Lineal Multiple

In [ ]:
#__ Cargar librerias
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import shapiro
from statsmodels.api import qqplot
from statsmodels.formula.api import ols
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
#___ Cargar la hoja de trabajo en un dataframe
reglin_m_df = pd.read_excel('datos_analisis_estadistico.xlsx', sheet_name = 'REG LINEAL M')
reglin_m_df.columns = ['ELEMENTO', 'EDAD', 'SUENO', 'IMC']
reglin_m_df

In [ ]:
#___ Prueba de linealidad VISUAL SEABORN
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,4))

ax1.scatter(x='EDAD', y='IMC', data=reglin_m_df)
ax1.set_facecolor("#343837")          # Inner color
ax1.figure.set_facecolor("#343837")    # Outer color
ax1.set_xlabel('EDAD', color='tomato')
ax1.set_ylabel('IMC', color='tomato')
ax1.tick_params(labelcolor='white')

ax2.scatter(x='SUENO', y='IMC', data=reglin_m_df)
ax2.set_facecolor("#343837")          # Inner color
ax2.figure.set_facecolor("#343837")    # Outer color
ax2.set_xlabel('SUENO', color='tomato')
ax2.set_ylabel('IMC', color='tomato')
ax2.tick_params(labelcolor='white')

plt.tight_layout()
plt.show()

In [ ]:
#___ Obtener los residuos para prueba de normalidad
modelo = ols('IMC ~ SUENO + EDAD', data=reglin_m_df).fit()
residuos = modelo.resid

In [ ]:
#___ Prueba de normalidad sobre los residuos (SCIPY)
prueba_norm = shapiro(residuos)
prueba_norm

In [ ]:
#___ Prueba de normalidad METODO VISUAL
qqplot(residuos, line='s')
plt.title("Grafica Q-Q del Modelo de Residuos")
plt.show()

In [ ]:
#___ Prueba de Homocedasticidad Breuch-Pagan (statsmodels)
prueba_homos = het_breuschpagan(modelo.resid, modelo.model.exog)
prueba_homos
###___ Output: (lagrange_multiplier_statistic (lms), p-value_lms, f-value, f-p-value)

In [ ]:
#___ Prueba residuos independientes.
###___ Se obtiene la homocedasticidad si los puntos estan distribuidos aleatoria y igualitariamente alrededor y=0
valores_ajustados = modelo.fittedvalues

fig, ax = plt.subplots(figsize=(6, 3))
ax.scatter(valores_ajustados, residuos)
ax.set_facecolor("#343837")        
ax.figure.set_facecolor("#343837")   
ax.axhline(y=0, color='tomato', linestyle='--', linewidth=1.5)
ax.set_xlabel('Residuos', color='tomato')
ax.set_ylabel('Valores Ajustados', color='tomato')
ax.tick_params(labelcolor='white')
plt.show()

In [ ]:
#___ Prueba analitica residuos independientes (DURBIN-WATSON)
###___ Interpretacion del estadistico Durbin-Watson:
###___ Cerca de 2: No auto-correlacion.
###___ Cerca de 0: auto-correlacion positiva.
###___ Cerca de 4: auto-correlacion negativa.
prueba_res_indep = durbin_watson(residuos)
prueba_res_indep

In [ ]:
#___ Prueba analitica de multicolinealidad
###___ Interpretacion del factor de inflacion de la varianza (VIF)
###___ Valores cercanos a 1 i indican que los predictores son independientes
###___ Valores entre 1 y 5 muestran una correlacion moderada que es en ocasiones aceptable
###___ Valores arriba de 10 señalan que hay una multicolinealidad problematica
datos_vif = pd.DataFrame()
datos_vif["Predictor"] = ['EDAD', 'SUENO']
datos_vif["VIF"] = [variance_inflation_factor(reglin_m_df[['SUENO', 'EDAD']], i) for i in range(reglin_m_df[['SUENO', 'EDAD']].values.shape[1])]
datos_vif

# # Bar Plot for VIF Values
# datos_vif.plot(kind='bar', x='Predictor', y='VIF', legend=False)
# plt.title('Variance Inflation Factor (VIF) by Feature')
# plt.ylabel('VIF Value')
# plt.show()

In [ ]:
#___ La recta ajustada al modelo de regresion lineal esta dada por los coeficientes
modelo.params

In [ ]:
#___ Valor P de la prueba de regresion lineal
print(modelo.summary())